# Computer Vision Workshop: Day 1

### Image processing → ensemble & margin classifiers

Over the next two days we'll go from raw pixels to trained classifiers, using plankton imagery from
the [Scripps Plankton Camera System](https://aslopubs.onlinelibrary.wiley.com/doi/full/10.1002/lom3.10394)
and [ZooScan](https://sites.google.com/view/piqv/), plus a terrestrial dataset
([Ohio Small Animals](https://lila.science/datasets/ohio-small-animals/)) later in the course.


## Getting started on the DSP

1. Connect to the NOC VPN
2. Open the DSP JupyterHub: [notebooks.noc.ac.uk/hub/spawn](https://notebooks.noc.ac.uk/hub/spawn)
3. From the Launcher, open a **Terminal**


## Clone the workshop materials

In the terminal:

> ```bash
> mkdir workshops
> cd workshops
> git clone https://github.com/NOC-OI/computer-vision-workshop.git
> cd computer-vision-workshop
> ```


## Set up the environment

Still in the `computer-vision-workshop` folder:

> ```bash
> conda env create -n cv-workshop -f environment.yaml
> ```

> ```bash
> python -m ipykernel install --user --name cv-workshop
> ```

- Takes a few minutes
- The `cv-workshop` kernel then appears in your Jupyter Launcher; select it for this notebook


## How this notebook works

- Presentation sections (like this one) give context. Code + exercise sections are the hands-on
  modules, unchanged from the standalone module notebooks.
- Run code cells with **Shift+Enter**.
- Code cells have a grey background; output appears directly below.
- Some exercises give you pseudo-code scaffolding to fill in yourself.


## Today's plan: "classical" computer vision

1. Image manipulation
2. Region finding (segmentation)
3. Feature extraction
4. Ensemble and margin classifiers

Each stage builds on the last, leading up to classifying an object from an image.


## What is computer vision?

- A subfield of AI: teach a computer to understand 2D data
- Called "vision" mostly because images are the common test case

**In scope for this workshop:**
- Supervised classification
- Feature selection
- Ensemble / margin classifiers
- Convolutional neural networks

**Out of scope:**
- Dataset construction & annotation best practices
- Unsupervised models
- Generative AI


## A few common computer vision tasks

- **Classification**: assign a single label to an image ("what is this?")
- **Object detection**: locate and label multiple objects within an image
- **Segmentation**: classify an image down to the pixel level

The same image can have many valid labels depending on the task: a photo of a giraffe could be
*giraffe*, *animal*, *outdoors*, or *Southern giraffe*, depending on what the classifier was built
to do.


## Classification, in practice

<img src="assets/slides/intro/classification_giraffe.jpg" width="420" alt="A giraffe looking directly at the camera">

*Giraffe? Animal? Outdoors? Southern giraffe? All correct; it depends on the task.*


## Segmentation, in practice

<img src="assets/slides/intro/segmentation_horse.png" width="600" alt="A photo of a running horse next to its pixel-level segmentation mask">

*Classifying every pixel: "horse" vs. "not horse."*


## Where computer vision shows up in ocean science

- **Acoustics**: classifying humpback whale vocalizations from spectrogram "images" (Google AI, 2018)
- **Imaging**: counting and identifying marine organisms from towed-camera and UAV imagery
  (Orenstein et al., 2025)
- **Ecology**: species identification from field photos, e.g. BioCLIP (Stevens et al., 2024, CVPR)


## Acoustics as images

<img src="assets/slides/intro/acoustics_spectrogram.png" width="620" alt="Spectrograms of humpback whale calls with a detected call highlighted in yellow">

*A spectrogram is just an image; the same CV tools apply (Google AI, 2018).*


## Detecting and segmenting marine organisms

<img src="assets/slides/intro/ocean_imaging_orenstein2025.jpg" width="700" alt="Four underwater imaging examples: benthic organism segmentation, deep-sea object detection, fish detection near a wreck, and jellyfish segmentation">

*Detection and segmentation across imaging platforms: benthic habitat, deep sea, wrecks, midwater
(Orenstein et al., 2025).*


## Species ID from field photos

<img src="assets/slides/intro/ecological_imaging_bioclip.jpg" width="620" alt="A plankton image alongside a bar chart of BioCLIP's predicted class probabilities, topped by Ciliate mix">

*BioCLIP's top prediction for this plankton image (Stevens et al., 2024, CVPR).*


## How do we produce a model?


### High-level steps to train a supervised CV model

1. Define the problem you want to solve
2. Find labeled data
3. Choose an architecture (model type) and a framework (software library)
4. Download model weights (if not training from scratch)
5. Split your data into training and validation sets
6. Train
7. Test, measure performance, and keep evaluating over time


### Choosing an architecture and framework

- Larger architectures take longer to train and run, so budget can rule some out
- A good starting point: whatever a similar published problem used
- Normal to try several architectures in parallel and compare
- Framework choice is often constrained by:
  - What your collaborators use
  - What your chosen architecture supports
  - Which documentation "clicks" for you


### What we'll focus on

**Today, classical machine learning:** hand-engineered features feeding **ensemble or margin
classifiers**.

- **Training:** draw features from labeled images, fit a classifier
- **Testing:** draw the same features from unseen images, check performance

**Tomorrow, convolutional neural networks:** learn features directly from images instead.


### A first look: convolutional neural networks

- Learn features directly from labeled images, no hand-engineering
- Simplest architectures are a series of filters
- Filter shapes and weights are learned during training
- Require input images to all be the same size

<img src="assets/slides/intro/cnn_diagram.png" width="700" alt="Diagram of a CNN: a copepod image passing through several convolutional feature-map layers narrowing toward an output">


### A word of caution: distribution shift

Every automated classifier assumes training and test data are drawn from the same distribution.
In practice, that's hard to guarantee:

- Organism distributions can change over time or space
- New classes (or noise) can appear; others can disappear
- Different instruments and conditions behave differently

Keep this in mind as we build and evaluate classifiers: how you split data, what counts as
"success," and how you'd notice a model has failed are all open questions we'll return to.


### Distribution shift, in practice

A ResNet classifier's counts of one plankton class, plotted against manual counts of the same
samples over three years:

<img src="assets/slides/intro/distribution_shift_example.png" width="560" alt="Plot of prevalence over time for a plankton class, comparing a ResNet classifier to manual counts, with a red box highlighting a period where they diverge">

- For most of the time series, including a bloom peak in mid-2016, the model tracks the manual
  counts closely
- In late 2017 (red boxes), the model starts consistently over-counting relative to the manual
  count
- The cause: novel-looking objects (inset photos) that resemble the target class closely enough to
  fool a model trained on the earlier distribution

(Orenstein et al., 2020)


## Modules 1–3: image processing & computer vision

Image processing is a subfield of signal processing that treats an image as a 2D signal:

- **Manipulation**: resizing, warping, other transforms (Module 1)
- **Filtering**: edge detection, region finding (Modules 2–3)

Many Photoshop-style features are built on exactly these ideas.


## Further reading

- *Digital Image Processing*, 4th ed., Gonzalez & Woods (Pearson)
- *Computer Vision: A Modern Approach*, 2nd ed., Forsyth & Ponce (Pearson)
- *Computer Vision: Algorithms and Applications*, Szeliski (Springer), free at
  [szeliski.org/Book](http://szeliski.org/Book/)


# 📘 Module 1: Image manipulation

**Goals:**
- Get comfortable manipulating images in Python
- Learn a few common image transformations
- Understand how transformations subtly change an image's appearance, which matters again later,
  once we feed images into deep nets


## Images are matrices of values

- A digital image is a matrix of numbers
- In an 8-bit gray scale image, each pixel's value (its "gray level") ranges from 0 (black) to
  255 (white)
- Some sensors have greater bit depth and represent gray levels more precisely

<img src="assets/slides/day1/gray_level_ramp.png" width="380" alt="Gray level ramp from 0 (black) to 255 (white)">


## Color images are 3D matrices

- One 2D layer ("channel") per color
- Each channel on its own is just a gray scale image

<img src="assets/slides/day1/channel_all_split_mod1_s23.png" width="620" alt="An RGB plankton image (928, 1736, 3) split into separate Red, Green, and Blue channels, each (928, 1736)">


## Affine transforms

Most transforms in this module are **affine transforms**: 2D transforms that map a pixel at one
point to another, while preserving parallel lines in the image.

- Translation
- Rotation
- Resizing (scaling)
- Warping


## The transform equation

Every affine transform can be written as the same matrix equation:

$$\begin{bmatrix} x_{new} \\ y_{new} \end{bmatrix} = A \times \begin{bmatrix} x_{orig} \\ y_{orig} \end{bmatrix} + B$$

- $A$ and $B$ take different forms depending on the transform
- e.g. $A$ = identity matrix, $B$ = a constant offset → pure translation

[Further reading on affine transforms](https://homepages.inf.ed.ac.uk/rbf/HIPR2/affine.htm)


## Perspective transforms

- More general than affine transforms
- Preserve straight lines, but *not* parallel ones
- Can digitally mimic the effect of zooming
- Covered at the end of this module

[Further reading on perspective warps](http://alumni.media.mit.edu/~maov/classes/comp_photo_vision08f/lect/08_image_warps.pdf)


The module below covers pixel indexing, gray scale conversion, and all of these transforms in
code, which you'll need later for data augmentation.

---


# Module 1 - Image Manipulation

To run all but the most basic commands in Python we need to import packages that contain most of the functionality.
Here we will introduce a few that are useful for image manipulation:

1. numpy is a matrix manipulation suite. It is akin to the basic processing package availble in MATLAB. 
2. matplotlib is a graphics toolbox for python. 
3. cv2 is OpenCV, an open-source computer vision library. OpenCV has lots of useful functions to manipulate and analyze images. We will make ample use of the tools in OpenCV

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import glob

Now all the packages are imported into the workspace. We can now use them inline below. 

First we will grab an image to play with. OpenCV has a really easy routine for it. After, we will display it with plt.

In [ ]:
# get the file path of the image in the directory
# glob is a library that impliments pathname pattern expansion in python. os is an operating system navigation package

# this command creates a list of all files with the letters SPC as the first characters in the current working directory. 
ptf = glob.glob(os.path.join(os.getcwd(), 'computer-vision-workshop/assets', 'SPC*'))

# We will grab the first item in the list
img = cv2.imread(ptf[0])

# now display it so we can see what we are working with
plt.imshow(img)

## Module 1 – Image manipulation and transformation

Before getting into the topic of edge detection and region finding, we will practice a few basic image manipulations. Most of these transformations are implemented for us in OpenCV. For now, the routines are mostly being explored to gain some intuition for working with images in Python. But these routines will eventually be useful for *data augmentation* for running deep nets.

First, we will check the data type.
A lot of errors in OpenCV and other image processing libraries occur because of type errors.
The data type is a property of the image and can be accessed with *img.dtype*.

In [ ]:
print(img.dtype)

There are many datatypes that could be useful and images are often convereted from type-to-type depending on the operation being preformed. *uint8* stands for "Unsigned integer" and is in the range of 0 to 255. 0 = black, 255 = white. 

Numpy matrices also make the min and max values easy to access:

In [ ]:
# print out the min and max values
print("The max pixel value is: ", str(img.max()))
print("The min pixel value is: ", str(img.min()))

### Module 1 – Pixels and values
Next, we can start messing with images by referencing particular pixels via their index. Most color images are treated as a 3-D matrix consisting of rows, columns and color chanels. 

The images provided for this tutorial from the Scripps Plankton Camera have three color channels. In OpenCV, images are loaded with colors in BLUE, GREEN, RED order.

To view an image in RGB convert it using cv2.cvtColor.

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)

Remember that in Python, indicies are referenced to 0. This is different from R and MATLAB that have all indicies starting at 1.

In [ ]:
# Print out the dimensions of the image
img.shape

In [ ]:
# Retrieve the color values at a particular pixel.
px_val = img[500, 800, :]
print("[B, G, R] values: " + str(px_val))

In [ ]:
# just look at the RED value
red_val = img[500, 800, 2]
print("The red channel value: " + str(red_val))

Say we want to examine a single color channel from our image 

In [ ]:
# copy the channel to a new array
img_red = img[:, :, 2]

# now see what it looks like
plt.imshow(img_red, cmap="gray")

Lastly, it is often convenient to convert a color image to gray scale. Again, OpenCV has a built in fuction for that.

In [ ]:
# covert to gray scale with OpenCV
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# give it a look
plt.imshow(img_gray, cmap='gray')

Note that both the *cv2.cvtColor* and *plt.imshow* each used optional arguements. For the gray scale converstion, the optional arguement is telling OpenCV how to interpolate the colors. Likewise, in the figure display, matplotlib needs the appropriate colorscale.

Lastly, sometimes it is useful to select a subsection of an image. This might come in handy for selecting a Region Of Interest from a larger frame or data augmentation

In [ ]:
# Select a small chunk of the gray sacle frame
img_sub = img_gray[400:800, 400:800]

# check it out
plt.imshow(img_sub, cmap='gray')

#### Module 1 – Practice
Grab a subregion of the original image with all the color channels. 

In [ ]:
# subregion of color image
clr_sub = img[400:800, 400:800, :]

# plot it
plt.imshow(clr_sub)

Now convert your subregion to gray scale, check the min and max pixel values, and plot it.

In [ ]:
# make your subregion gray scale
clr_sub_gray = cv2.cvtColor(clr_sub, cv2.COLOR_BGR2GRAY)

# print out the min and max values
print("The max pixel value is: ", str(clr_sub_gray.max()))
print("The min pixel value is: ", str(clr_sub_gray.min()))

# plot it
plt.imshow(clr_sub_gray, cmap='gray')

## Module 1 – Transforms
### Module 1 – Resizing
The most common type of transform is resizing. OpenCV has a builtin function to do this: 

In [ ]:
# resize the original image to make it half as large

# first save grab the dimensions for use in the function
hh, ww = img.shape[:2]

# then halve the values
new_hh = hh/2
new_ww = ww/2

# print them out
print("The new height will be: ", str(new_hh))
print("The new width will be: ", str(new_ww))

Notice that these numbers are float (ie decimals). OpenCV requires that indices are integers. We use Python built-in conversion command

In [ ]:
# the second arguement in resize tells it what to do with height
img_resize = cv2.resize(img, (int(ww/2), int(hh/2)))

# plot it
plt.imshow(img_resize)

Be aware that the image resize function **does not** preserve the aspect of the image. This becomes important when running deep nets that require input images to have the same dimensions. After transformation, the input data will not retain aspect information that may be important for classification.

In [ ]:
# force the original images to be a square matrix
img_square = cv2.resize(img, (512, 512))

# plot it
plt.imshow(img_square)

The effect may or may not be important depending on your application.

### Module 1 – Image warping
Most transforms in OpenCV are implimented as either an affine or perpective warps. Affine transformations preserve parallelism between the original and modified image -- a set of parallel lines in the original image will remain parallel in the new one. Perpective transforms only preserve straight lines between the original and transformed images.

#### Module 1 – Affine transformations set up
Use these transformations for things like translating, rotating, or shearing the image. All of these transformations require generating a transformation matrix that dictate the mathematical instructions for the warping.   

To translate an image 50 pixels down and 200 pixels to the right

In [ ]:
# this is the translation matrix. The desired amount of shift is placed in the final column of the matrix
mm = np.float32([[1, 0, 200], [0, 1, 50]])
print("Translation matrix: ")
print(mm)

In [ ]:
# now do the warping wiht warpAffine()
img_translate = cv2.warpAffine(img, mm, (ww, hh))

# plot it
plt.imshow(img_translate)

Rotating the image about a point requires a different matrix that OpenCV will set up for you with *getRotationMatrix2D*. The first argument defines the point about which to do the rotations. For example, to rotate the image 45 degrees about the center:

In [ ]:
# get the rotation matrix
mm = cv2.getRotationMatrix2D((ww/2, hh/2), 45, 1)

# warp it
img_rot = cv2.warpAffine(img, mm, (ww, hh))

# plot it
plt.imshow(img_rot)

#### Module 1 – Perspective transforms
Perspective warping is good for mimicing the effect of zooming in or out of an image. To produce the behavior, you need to provide *getPerspectiveTransform* with two sets of four points.

In [ ]:
orig_pts = np.float32([[50, 50], [52, 800], [1500, 42], [1550, 750]])
dst_pts = np.float32([[0, 0], [0, 750], [1450, 0], [1450, 750]])

mm = cv2.getPerspectiveTransform(orig_pts, dst_pts)

img_pres = cv2.warpPerspective(img, mm, (1450, 750))

plt.imshow(img_pres)

#### Module 1 – Practice

In [ ]:
# Flip the test image (rotate it 180 degrees about the middle)
mm = cv2.getRotationMatrix2D((ww/2, hh/2), 180, 1)

# warp it
img_flip = cv2.warpAffine(img, mm, (ww, hh))

# plot it
plt.imshow(img_flip)

In [ ]:
# now translate the rotated image up 50 pixels and 100 to the left
mm = np.float32([[1, 0, -100], [0, 1, -50]])

img_flip_tran = cv2.warpAffine(img_flip, mm, (ww, hh))

# plot it
plt.imshow(img_flip_tran)

# 📘 Module 2: Segmentation and region finding

Segmentation selects the objects of interest out of a full frame.

- **Uniform background** (e.g. plankton microscopy): comparatively easy
- **Structured background** (e.g. benthic habitat): far more complex, still an active research
  problem


## Filtering as convolution

A small kernel slides across the image; each position's output is a weighted combination of the
pixels underneath it.

<img src="assets/slides/day1/convolution_filter.gif" width="420" alt="Animation of a 3x3 mean filter kernel sliding across an image and producing one output value per position">

*Courtesy of University of Calgary.*


## Morphological clean-up

A structuring element sweeps over the mask; any pixel it touches gets added to the foreground,
one of the "clean-up" steps used after thresholding.

<img src="assets/slides/day1/dilation_animation.gif" width="300" alt="Animation of morphological dilation: a cross-shaped structuring element sweeping over a grid, adding a pixel wherever it touches an existing foreground pixel">

*Dilation, using a cross-shaped structuring element. Courtesy: Poonam Kshirsagar.*


## Module 2 – When backgrounds get complicated

Structured backgrounds (benthic habitat imagery is the classic example) break the simple
threshold-and-clean pipeline.

<img src="assets/slides/day1/benthic_coral_example.png" width="420" alt="Underwater photo of a structured coral reef habitat, illustrating a complex segmentation background">

*No single threshold separates "coral" from "not coral" here (Steffens et al., 2019).*


## Strategies for complex backgrounds

- **Point annotation**: randomly sample points, classify a small neighborhood around each to
  estimate coverage (e.g. [CoralNet](https://coralnet.ucsd.edu/); Steffens et al., 2019)
- **Stereo image pairs**: fully automated segmentation (King et al., 2018), though even the best
  method only got ~66% of pixels correct across all classes
- **[deep-segments](https://github.com/andrewcking/deep-segments)**: King et al.'s tool using ML
  to speed up human ground-truthing


## How well does automated segmentation work?

<img src="assets/slides/day1/king_et_al_segmentation.png" width="700" alt="Comparison of ground truth coral segmentation against FCN8s, Dilation8, DilationMod, and DeepLab v2 automated methods">

*Ground truth vs. four automated methods on the same benthic image (King et al., 2018); even the
best only partially agrees with ground truth.*

---


*Starting Module 2 fresh: the cell below clears all variables from the previous module before continuing, matching how these notebooks behaved when run standalone in their own kernel.*

In [ ]:
%reset -f

# Module 2 - Image Segmentation and Region Finding

Before the popularization of neural networks, engineers and scientists spent lots of time developing routines to crop out regions of an images. This process of image segmentation and region finding is the first step to classifying images with margin and ensemble classifiers. Once the regions of interest are detected, features can be extracted to train, test, and apply a classifier. 

These techniques are useful for preprocessing data raw data and generating image metrics that preserve the original, physical scale of the data. It also provides a helpful baseline to compare against deep methods.

As with any python application, we first import the necessary libraries.

In [ ]:
import numpy as np
import cv2
import skimage
from skimage import filters, morphology, measure, color
from scipy import ndimage
import sys
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.patches as ptch

We have added a new library to work with: skimage, short for Scikit-Image. This is another image processing toolbox that adds additonal functionality to OpenCV.

To start with, we will pull up a raw SPC image. This is what is directly captured on the sensor *in situ*.

In [ ]:
# get the file path of the image in the directory
ptf = glob.glob(os.path.join(os.getcwd(), 'computer-vision-workshop/assets', 'SHRINK-SPC*'))
print(ptf[0])

# We will grab the first item in the list
img = cv2.imread(ptf[0])

# change to 

# now display it so we can see what we are working with
plt.imshow(img)

This does not look like much. Indeed, most of the full frame image is empty space. But take a closer look. Try taking a subimage from the full frame. Constrain the height between 550 and 700  and the width between 400 and 600.

In [ ]:
img_sub = img[550:700, 400:600]
plt.imshow(img_sub)

A human can go through and grab everything out of the frame in this manner, but it would be time consuming. Instead, we can use edge detection to find all the objects. 

## Module 2 – Region finding

There are many ways to find regions in an image. The specific method choosen for your data very much depends on the type of images and the background. We will just explore a few here.

It is important to note that none of these are completely fool proof. They all require some amount of human effort to emprically set parameters that dictate the behavior of the algorithm. It is important to test the fidelity of the code under many different conditions to ensure that it is behaving as expected.

### Module 2 – Thresholding

If the image background is uniform enough, setting a binary threshold to find pixels above a certain value might be sufficent. 

In [ ]:
# first make a copy of the full image as a gray scale image
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# now look at some of the image parameters. here we will use numpy to compute a few things
print("the max px: ", str(np.max(img_gray)))
print("the min px: ", str(np.min(img_gray)))
print("the mean px: ", str(np.mean(img_gray)))

Given that most of the image is that minimum value or close to it, the mean might be an effective filter. 

In [ ]:
# use numpy to select the pixels and scale it
img_thresh = np.where(img_gray < np.mean(img_gray), 0., 1.0)

# this time plot the whole image and the subregion next to each other
fig, ax = plt.subplots(1, 2)  # subplots allow defining two sets of axes
ax[0].imshow(img_thresh, cmap='gray') # they can then be accessed numerically
ax[1].imshow(img_thresh[550:700, 400:600], cmap='gray')
plt.show()

The mean threholded image is kind of noisy. One option is to empirically pick several other threshold values and see how they work. Fortunately, there is an algorithmic approach. 

Otsu's method searches for a threshold in the image by examining it's intensity histogram.

In [ ]:
# Make an intensity histogram.
# ravel flattens the array, 256 is the number of bins, and [0, 256]
plt.hist(img_gray.ravel(), 256, [0, 256])
plt.show()

This displays the number of pixels in each bin. Clearly the most common bin is zero (ie black). This is akin to an intensity histogram you might use in Photoshop or iPhoto to mess with an images color. Zoom in to see more structure.

In [ ]:
# zoom in on the intensity plot
plt.hist(img_gray.ravel(), 256, [0, 256])
plt.ylim([0, 10000])
plt.show()

Now we see that there are two peaks in the gray scale image. Otsu's method attempts to find a threshold that minimizes the variance of the pixels on either side of the boundary.

In [ ]:
# use the skimage implimentation of Otsu's method
thresh = filters.threshold_otsu(img_gray)
print("Otsu threshold: ", str(thresh))

# now plot it on the intensity histogram 
plt.hist(img_gray.ravel(), 256, [0, 256])
plt.ylim([0, 10000])
plt.axvline(x=thresh, color='r')
plt.show()

This might be more effective than using the mean. 

In [ ]:
img_otsu = np.where(img_gray < thresh, 0., 1.0)

# plot the whole image and the subregion next to each other
fig, ax = plt.subplots(1, 2)  # subplots allow defining two sets of axes
ax[0].imshow(img_otsu, cmap='gray') # they can then be accessed numerically
ax[1].imshow(img_otsu[550:700, 400:600], cmap='gray')
plt.show()

While not perfect, Otsu's threshold helped element a lot of the small noise particles. 

Generally speaking, thresholding will discard a good amount of information. It works pretty well on SPC images because the foreground pixels are so distinct from the dark background. But you can see how difficult it would be to select a value that would grab all the stuff you are interested in a more complex image. 

### Module 2 – Filtering

Another good option for selecting regions in an image is *filtering*. An image filter is a sliding window that is dragged across an image to perform an operation in a neighboorhood around every pixel. A 3x3 median fitler, for example, computes the median value in a 3x3 window around the central pixel. In general, filters have odd numbered dimensions.

Akin to acoustics, filtering images can be described mathematically as a 2D convolution. That means the computer can cast the filtering operation as a multiplication in frequency space, rather than iteratively computing a value at every index. The details go beyond the scope of the tutorial.

There are many different operations that are done with filtering. In fact, convolutional neural networks make use of filters in the feature extraction phase. Here, we will use them to find edges.

#### Module 2 – Edge detection
Edges in images can be modeled as high frequency component of the image matrix. In other words, edges tend to be sharp discontinuities in pixel values. In the image we are working with, the pixels containing the plankton are bright and the background is dark. Where the pixels transition from light to dark will show up as an edge in the image.

We can exploit this to search for edges. First, try out a *Laplacian filter* -- a filter that computes the second derivative within the window. 

In [ ]:
# Use a laplacian filter from OpenCV
img_laplace = cv2.Laplacian(img_gray, cv2.CV_16UC1, ksize=1)

# plot the whole image and the subregion next to each other
fig, ax = plt.subplots(1, 2)  # subplots allow defining two sets of axes
ax[0].imshow(img_laplace, cmap='gray')
ax[1].imshow(img_laplace[550:700, 400:600], cmap='gray')

The second arguement in the Laplacian command specifies the image depth, or data type of the output. We set it to uint16 to increase the percision of the computation. This is not always necessary, but it helps for a mostly empty image. *ksize* specificies the kernel size. ksize=1 gives us a kernal that look likes this:

$$\left[
\begin{matrix}
0 & 1 & 0 \\
1 & -4 & 1 \\
0 & 1 & 0 
\end{matrix}
 \right] $$
 
The 3x3 neighborhood around each pixel is multiplied by this matrix.The resulting values are all added together to get the output for the index at the center. Note if k > 1 e.g. 3,5,7 we get a larger kernel but should always be an odd number. 

This output looks better at a birds eye view. Take a closer look the gelatinous region. 

In [ ]:
# select subregion and plot
plt.imshow(img_laplace[550:700, 400:600], cmap='gray')

We have found the outline of all the objects of the in the image. And we are starting to resolve some of the structure in the body of the organism itself.

#### Module 2 – Canny edge detector

The Canny edge detector is a multistage algorithm for edge detection written by John Canny in 1986. It works very well in many cases and is a common element of an image processors toolbox. There are 4 steps: 

1. Noise reduction -- generally done with a Gaussian smoothing filter. Basically, make the image a little blurry. 
2. Finding intensity gradients with a filter similar to the Laplacian.
3. Non-maximal suppresion forceses the edges to be thin. This stage outputs a binary images with the range [0 255].
4. Hysteresis thresholding determines which edges are real. This is done based on the minimum and maximum values the engineer gives the algorthm. A value above the max is sure to be an edge. A value below the minimum is sure *not* to be an edge. In between the max and the min, the algorithm checks to see if a line segement connects to a sure edge.

Setting the both the max and minimum values for the hysteresis thresholding high makes the edge detector more conservative. These thresholds need to be tinkered with empirically.

In [ ]:
# run canny
img_canny = cv2.Canny(img_gray, 150, 225)

plt.imshow(img_canny, cmap='gray')

In [ ]:
plt.imshow(img_canny[550:700, 400:600], cmap='gray')

Try changing the max and min values of the threshold to see what you get. 

### Module 2 – Extracting regions

Pulling out smaller regions from an image requries locating the desired objects. There are many algorithms implimented in both OpenCV and skimage to find region in an image. Here, we will illustrate the use of skimage's *label* routine. 

*label* uses connected component analysis to locate and label regions in an image. It checks each pixel to see how many neighboring pixels are in the foreground. Once it crawls all the pixels in a region, it gives the region a numeric label. The process repeats itself until all pixesl have been considered. 

To aid the process, it is good practice to use a morphological operators to connect edges and complete outlines. Morhpological opening and closing drag a *structuring element* -- a predefined shape used to probe foreground regions in a binary image.

Here we will use morphological closing. This routine will close small dark spots in otherwise complete objects. For this example, let's work with the Otsu thresholded image. We will use a square structuring element 5 pixel edges. 

In [ ]:
img_close = morphology.closing(img_otsu, morphology.square(5))

# plot the whole image and the subregion next to each other
fig, ax = plt.subplots(1, 2)  # subplots allow defining two sets of axes
ax[0].imshow(img_close, cmap='gray')
ax[1].imshow(img_close[550:700, 400:600], cmap='gray')

Now compare the original Otsu mask with the closed one.

In [ ]:
# plot the whole image and the subregion next to each other
fig, ax = plt.subplots(1, 2)  # subplots allow defining two sets of axes
ax[0].imshow(img_otsu[550:700, 400:600], cmap='gray')
ax[1].imshow(img_close[550:700, 400:600], cmap='gray')

With these complete masks we can now use skimage's region labeling. It will decide if a pixel belongs in a region by checking its connectivety using 8 neighbors. Consider the following grid:

$$\begin{matrix}
0 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 0 
\end{matrix}$$

The pixel in the center is one, but all its neighbors are zero. Since it is not connected to anything else, it will be its own regions.

$$\begin{matrix}
1 & 0 & 0 \\
1 & 1 & 0 \\
1 & 0 & 0 
\end{matrix}$$

The pixel in the center of this grid, however, will be a part of the region defined by the neighbors to it's left. When using 8-neighbor connectivity, any value of 1 in the pixels around the center will register as a contiguous region. 

In [ ]:
# pass the label routine the closed image to register connected regions.
label_img = morphology.label(img_close, background=0)
lab_img_color = color.label2rgb(label_img, image=img_gray)

# plot the whole image and the subregion next to each other
fig, ax = plt.subplots(1, 2)  # subplots allow defining two sets of axes
ax[0].imshow(lab_img_color, cmap='gray')
ax[1].imshow(lab_img_color[550:700, 400:600], cmap='gray')

The colors correspond to different labeled regions. We can now feed this to labeled image to skimages *regionprops*, a routine that computes lots of information about a region based on the pixels inside it. For now, we will just use it compute the area -- the total number of pixels in the labeled region -- and the dimensions of a bounding boxes. 

In [ ]:
# The last arguement is a list of features to pull out of each region
props = measure.regionprops(label_img, img_gray, ['Area', 'BoundingBox'])

# The length of this vector correspond to the number of regions found
print("The number of regions is: ", str(len(props)))

That is a lot! regionprops grabbed all the little bits of detritus in the image. We can filter by area to only examine the biggest regions in the original image. 

In [ ]:
# first open up the original gray scale image as a background 
fig, ax = plt.subplots()
ax.imshow(img_gray, cmap = 'gray')

# initalize an empty vector to store all the bounding boxes
prop_out = []

# set the area threshold
area_thresh = 500

# iterate through the regions
for prop in props:
    
    # only select those with an area bigger than the threshold area
    if prop.area > area_thresh:
        
        # save a list of the big ones
        prop_out.append(prop.bbox)
        
        # Bounding box returns the coordinates of the miniumn and maximum row and column locations as:
        # [min_row, min_col, max_row, max_col]
        # We can use those values to generate a box in the image.
        rect = ptch.Rectangle((prop.bbox[1], prop.bbox[0]), prop.bbox[3] - prop.bbox[1], prop.bbox[2] - prop.bbox[0],
                          fill=False, edgecolor='red', linewidth=2)

        ax.add_patch(rect)
        
plt.show()

# state how many regions fit the critieria
print(str(len(prop_out)), " regions over ", str(area_thresh), " pixels")

We can now iterate through each of these and crop out the regions in side the bounding box.

In [ ]:
# iterate through the regions and cut out the pixels from the original image. 
# first add some padding around the region to make sure the bounding box doesn't cut anything off. 
# here we will use half the width, effectively doubling the size of the boundary. 

# first initalize a dictionary to store the output. This will allow us to store arrays of different sizes. 
roi_out = dict()

# define an flag to count off the regions
flag = 0

for bbox in prop_out:
    
    # first, get the height and width of the box
    width = bbox[3] - bbox[1]
    height = bbox[2] - bbox[0]

    # now we will define the upper left corner of the box
    # make sure the values are integers for indexing with np.floor
    yy = bbox[1] - np.floor(width/2)
    xx = bbox[0] - np.floor(height/2)
    
    # force xx and yy to be integers
    xx = int(xx)
    yy = int(yy)
    
    # if either of the values are negative, force them to zero
    # this makes sure we get regions at the edge
    if xx < 0:
        xx = 0
    
    if yy < 0:
        yy = 0
        
    # now extract the region
    roi_temp = img_gray[xx:xx+2*width, yy:yy+2*height]
    
    # create a dictionary key with an f-string
    out_str = f"roi_{flag}"
    
    # save it out
    roi_out[out_str] = roi_temp
    
    # make sure to increase the value of the string
    flag += 1
    
print('Created ', str(flag), ' rois')

Now, loop through the dictonary to see what the results look like. 

In [ ]:
# loop over all the keys (roi names) in the dictonary
for kk in roi_out.keys():
    # create a new figure
    plt.figure()
    
    # turn off the axis numbers to make it a bit more readable
    plt.xticks([])
    plt.yticks([])
    
    # label each ROI 
    plt.title(kk)

    # show it in gray scale
    plt.imshow(roi_out[kk], cmap='gray')
    plt.show()

These ROIs can then be saved iteratively as their own files. This is what is done on the onboard computer of the SPC and in other plankton imaging instruments. This effectively cuts down on the amount of data stored. 

Other more optically dense types of image, such as benthic images, require more involved technqiues for image segmentation. Edge detection might be effective if the environment is sparse enough. Othewise, texture-based segmentation methods might be effective. And, or course, neural network based region finding might be the best bet. 

# 📘 Module 3: Feature extraction

- Once objects are selected, we collect measurements about them
- A form of **dimensionality reduction**: thousands of raw pixels → a compact vector of metrics,
  `[x1, x2, x3, ...]`
- Which metrics work best takes trial and error


## Morphological features

Shape information: major axis, minor axis, equivalent spherical diameter, solidity, Hu moments...

<img src="assets/slides/day1/morphology_axes.png" width="500" alt="A copepod with its major axis drawn as a line and equivalent spherical diameter drawn as a circle of the same area">

*Major axis (line) and equivalent spherical diameter: a circle with the same area as the region
(green).*


## Texture features: GLCM

The Gray Level Co-occurrence Matrix summarizes how often pairs of pixel values co-occur at a given
angle and distance.

<img src="assets/slides/day1/glcm_angles.png" width="320" alt="Diagram of the eight GLCM angles (0-315 degrees) radiating from a central pixel">


## From one region to a feature matrix

- Each region → a vector of numbers
- Do this for every region, across every image → a feature **matrix** (rows = images, columns =
  features)
- This is exactly the input Module 4's classifiers expect

<img src="assets/slides/day1/feature_matrix.png" width="620" alt="Several organism images each mapped to a row of a feature matrix, indexed by image (rows) and feature (columns)">

---


*Starting Module 3 fresh: the cell below clears all variables from the previous module before continuing, matching how these notebooks behaved when run standalone in their own kernel.*

In [ ]:
%reset -f

# Module 3 - Feature extraction

To train and apply ensemble or margin classifiers like random forests or support vector machines, features must be measured from the images. Feature extraction, like region finding, requires a substantial amount of engineering and time and tweaking for a particular dataset.

Feature extraction routines assume that candidate regions have already been identified. In this module we will extract features from an example ROI from the SPC dataset. 

In [ ]:
import numpy as np
import cv2
import skimage
from skimage import filters, morphology, measure, color, feature
from scipy import ndimage, interpolate
import sys
import glob
import os
from math import pi
import matplotlib.pyplot as plt
import matplotlib.patches as ptch

In [ ]:
# get the file path of the image in the directory. here grab the diatom chain 
ptf = glob.glob(os.path.join(os.getcwd(),'computer-vision-workshop/assets', 'SPC*'))

# We will grab the first item in the list
img = cv2.imread(ptf[0])

# check to make sure we got the right thing
plt.imshow(img)

This region is one that we might have extracted using techniques for the previous module. Like when find the regions, we need to generate a binary mask to tell the computer what to focus on.

## Module 3 – Get a binary mask

To start the mask we will use a new edge detector called a Scharr filter. The Scharr filter is an operator similar to Canny that searchers for edges in the intensity image. It is well suited to finding high frequency edges.

In [ ]:
# make the image gray
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# compute the edges
edges_mag = filters.scharr(img_gray)

# see what it looks like
plt.figure()
plt.xticks([])
plt.yticks([])
plt.imshow(edges_mag, cmap='gray')

The Scharr filter returns gray scale values. Let's make it binary a setting a threshold that is 3 times the median value in the array. This makes our mask more apt to retain edges that using something like Otsu's method.

In [ ]:
edges_med = np.median(edges_mag)
edges_thresh = 3*edges_med
edges = edges_mag >= edges_thresh

# see what it looks like
plt.figure()
plt.xticks([])
plt.yticks([])
plt.imshow(edges, cmap='gray')

That doesn't look too bad. But we want to fill in the boundary and select only the largest one for processing. To start with, we can use morphological operations to seal up some of those holes. 

In [ ]:
# these will fill in some of the holes
edges = morphology.closing(edges, morphology.square(3))
filled_edges = ndimage.binary_fill_holes(edges)

plt.figure()
plt.xticks([])
plt.yticks([])
plt.imshow(filled_edges, cmap='gray')

The hole filling has expanded the mask a bit. The region growing effectively got rid of the holes, but bloated the mask. Eroding, another morphological operation, will shrink it down again. 

In [ ]:
img_close = morphology.erosion(filled_edges, morphology.square(3))

plt.figure()
plt.xticks([])
plt.yticks([])
plt.imshow(img_close, cmap='gray')

It also got rid of some of the smaller noise. We can now use skimages *label* routine to find all the connected regions in the frame. 

In [ ]:
# pass the label routine the closed image to register connected regions.
label_img = morphology.label(img_close, background=0)
lab_img_color = color.label2rgb(label_img, image=img_gray)

# plot the whole image and the subregion next to each other
plt.imshow(lab_img_color, cmap='gray')

The yellow boundardy encompasses the diatom chain and gets most of the spines. Before analyzing it, we need to select only the largest boundary in the frame. Regionprops will extract this information.

In [ ]:
# use region props. But this time, have it retain all the properties it can measure
props = measure.regionprops(label_img, img_gray)
max_area = 0
max_area_ind = 0
for f in range(0,len(props)):
    if props[f].area > max_area:
        max_area = props[f].area
        max_area_ind = f

ii = max_area_ind

# now just display that area with the bounding box to make sure it got the right one.

# this selects only the pixels in the labeled image that are in the region with the biggest area
bw_mask = (label_img) == props[ii].label

plt.figure()
plt.xticks([])
plt.yticks([])
plt.imshow(bw_mask, cmap='gray')

And to actually see the masked image, simply multiply the original array by the binary mask.

In [ ]:
img_masked = img_gray * bw_mask

plt.figure()
plt.xticks([])
plt.yticks([])
plt.imshow(img_masked, cmap='gray')

This will be region that from which all the features will extracted. 

## Module 3 – Region properties

There are many types of metrics that can be used for classification. We will touch on two sets in particular: morphology and texture. All of these features will be saved to a feature vector that will be saved for later use. 

In [ ]:
# create an empty feature vector to store everything
img_features = []


### Module 3 – Morphology

Morphological features describe the shape of the object. Some are familiar, such as the length, width, or area. Others might be a bit more exotic like the convex hull and eccentricity. 

regionprops has already computed a lot of this for us. First, let's look at what it got for the major and minor axes.

In [ ]:
# grab just that biggest region
prop = props[ii]

# the center and orientation of the region have been measures. That will be used to display the axes
yy, xx = prop.centroid
angle = prop.orientation

# the measurements of the major and minor axis
maj_ax = prop.major_axis_length
min_ax = prop.minor_axis_length

# compute the coordinates of the line segment ends
x1 = xx + np.cos(angle)*0.5*maj_ax
y1 = yy - np.sin(angle)*0.5*maj_ax
x2 = xx - np.sin(angle)*0.5*min_ax
y2 = yy - np.cos(angle)*0.5*min_ax

# plot it
fig, ax = plt.subplots()
ax.imshow(img_masked, cmap='gray')

# render the major axis from the centriod
ax.plot((xx, x1), (yy, y1), 'r-', linewidth=2)

# render the minor axis from the centroid
ax.plot((xx, x2), (yy, y2), 'r-', linewidth=2)

ax.set_xticks([])
ax.set_yticks([])
plt.show()

The major and minor axes, while not perfect, give a reasonable representation of the images. 

It is often convenient to compute *invariant* features, ones that are uneffected by changes in scale, rotation, or translation. Saving such features increases the likelihood that the future machine classifer will be able to recognize another diatom chain. 

The aspect ratio is insenstive to scale. In the case of the diatom chain, it will indicate that the ROI is long and skinny. Aspect ratio is unitless and scales between 0 and 1. 0 indicates a line, 1 would be a circle.

In [ ]:
aspect = prop.minor_axis_length/prop.major_axis_length

print("the aspect is: ", str(aspect))

# add it to the feature vector
img_features.append(aspect)

There are several area ratios that contain useful information. The first makes use of the convex hull of the object. The convex hull is the smallest convex polygon that contains all the pixels in a binary mask. We can see what a hull looks like using skimage.

In [ ]:
# compute the convex hull from the mask
hull = morphology.convex_hull_image(bw_mask)

# now plot the original mask next to the convex hull
fig, ax = plt.subplots(1, 2)

ax[0].imshow(bw_mask, cmap='gray')
ax[0].set_xticks([])
ax[0].set_yticks([])

ax[1].imshow(hull, cmap='gray')
ax[1].set_xticks([])
ax[1].set_yticks([])

The ratio of the area of the mask to the convex hull is indicative of how spiny an object is. region_props prints this as the solidity

In [ ]:
# set the data type to float to ensure we get a decimal out
sol = prop.solidity

# add it to the feature vector
img_features.append(sol)

The ratio of the area inside the region to the perimeter is also indicative of the spininess of the object. It is in essence a surface area-to-volume metric.

In [ ]:
area3 = prop.area.astype(np.float64)/(prop.perimeter*prop.perimeter)

# add it to the feature vector
img_features.append(area3)

regionprops computes the extent of the image as a ratio of the pixels in the regions to the pixels in the bounding box. This is another indicator of the how solid the object is.

In [ ]:
area2 = prop.extent

# add it to the feature vector
img_features.append(area2)

There are several other automatically features that regionprops computes. We will not cover them in any detail here. Some may or may not be useful depending on the context.

In [ ]:
# save the area
area = prop.area

# the eccentricity of an ellipse with the same second-moments as the masked region
ecc = prop.eccentricity

# diameter of a circle with the same area as the object
esd = prop.equivalent_diameter

# the euler number is 1 minus the number of holes in the object. For this diatom chain, it will be 1.
en = prop.euler_number

# add these to the feature vector
img_features.extend([area, ecc, esd, en])

Image moments are weighted averages of the image's pixel intensities. Hu moments are a set of 7 particular moements that approximately invarient across common image transformation like translation, scaling, and rotation. That means that the ROI can be warped in several ways and retain the same Hu moments. This property is very useful for classification.

In [ ]:
# add the hu moments to the feature vector
img_features.extend(prop.moments_hu)

### Module 3 – Texture

To this point, the features only contain information about the shape of the object. We can extract information about the texture using the intensity values in the masked image. 

In [ ]:
# make a histogram of the pixel intensities assuming the image has gray scale values between 0 and 256
img_hist = np.histogram(img_masked, 256)

# numpy's histogram returns the bin sizes, too. only look at the counts in each gray level
img_hist = np.asarray(img_hist[0]).astype(np.float64)

# since most of the image is black and we only care about the stuff in the region, set the bin for black pixels=0
img_hist[0] = 0

# normalize to scale it from zero to one
img_hist = img_hist/img_hist.sum()

plt.bar(range(0, 256),img_hist)

This histogram is the probability distribution of gray scale values in our ROI. We can now compute some feature from this PDF that are indicative of texture. 

In [ ]:
# a list of the normalized pixel values from 0 to 256.
vec = np.arange(0, len(img_hist)).astype(np.float64) / (len(img_hist) - 1)

# get the bins with nonzero number of pixels in them.
ind = np.nonzero(img_hist)[0]

# mean grey value
mu = np.sum(vec[ind] * img_hist[ind])

# variance in the gray scale values
var = np.sum((((vec[ind] - mu)**2) * img_hist[ind]))

# standard deviation of the gray scale values
std =  np.sqrt(var)

# contrast - a number indicating the difference between a pixel it's neighbors over the whole image
cont = 1 - 1/(1 + var)

# 3rd moment. A metric that indicates the distribution of gray values. 
# A high 3rd moment indicates that there are more bight pixels. Low 3rd moment says the opposite. A 3rd moment of zero
# means there is a roughly equal distribution of light and dark pixels
thir = np.sum(((vec[ind] - mu)**3)*img_hist[ind])

# Uniformity - how flat the intensities are. If the image a single value, uniformity=1
uni = np.sum(img_hist[ind]**2)

# Entropy is a measure of randomness in the pixel intenisties. 
ent = - np.sum(img_hist[ind] * np.log2(img_hist[ind]))

# add them all the feature vector
img_features.extend([mu, var, std, cont, thir, uni, ent])

These metrics computed from the histogram of intensities contain a great deal of information regarding the gray levels in the ROI. But they contain no information about the relative positions of the gray levels. That is, it might tell us that there is a repeating pattern, but not what the pattern is. 

The last set of texture metrics we will add to these feature vectors is thus the gray level coocurrence matrix (GLCM). The GLCM contains the probability of pixel pairs having the same value at a defined distance and orientation from each other. 

skimage's *graycomatrix* will compute such a matrix at a set of defined distances and orientations across a whole image. It will return a 4D matrix with the value at each combination of distances and orientation at every coordinate in the region.

This matrix is too big to save as a feature and probably contains redundent information anyway. Using skimage's *greycoprops* will compute summary statistics at each pair of distances and angles. Here we will tell it to comptute 4 summary statistics:

1. contrast - the sum of squared differences between the pixel pairs
2. dissimilarity - sum of the absolute differences between the pixel pair
3. energy - the square root of the sum of the squared probabilities of each pair.
4. correlation - the of each pair of values at the distance and angle

In [ ]:
# first, define the distances to use (in pixels)
dist = [1, 2, 4, 16, 32, 64]

# then define the angles (in radians)
ang = [0, pi/4, pi/2, 3*pi / 4]

# compute the matrix from the masked image
pp = feature.graycomatrix(img_masked, distances = dist, 
                         angles = ang, normed = True)

# define an output matrix for the metrics
grey_mat = np.zeros([24,2]) 

# set a flag to move the index of the output matrix
flag = 0

# tell skimage which metrics to compute and iterate over them
grey_props = ['contrast', 'homogeneity', 'energy', 'correlation']
for name in grey_props:
    
    # actually compute the features
    stat = feature.graycoprops(pp, name)
    
    # take the mean and standard deviation of the metrics to further compress them
    grey_mat[flag:flag+6,0] = np.mean(stat,1)
    grey_mat[flag:flag+6,1] = np.std(stat,1)
    flag += 6

# Add to feature Vector
img_features.extend(grey_mat[:,0])
img_features.extend(grey_mat[:,1])

print("total features: ", str(len(img_features)))

We have thus converted our diatom ROI into a vector of 72 numbers. When preparing to train and test an ensemble or margin classifier, all the labeled images would be run through such a routine. Indeed, the training and test sets for the next module have already been processed in this way.

There are a multitude of possible useful features for such things. Settling on which features are most appropriate for your application takes time and testing. 

# 📘 Module 4: Ensemble and margin classifiers

Finally, machine learning! Using the metrics from Module 3, we'll train two types of classifiers:

- **Support Vector Machine** (a *margin* classifier)
- **Random Forest** (an *ensemble* classifier)

*(Further reading: Pattern Classification, 3rd ed., Duda, Hart & Stork, Wiley-Interscience.)*


## Module 4 – Building intuition: fruit fly or porcupine?

Imagine classifying images into porcupine, fruit fly, or fish:

<img src="assets/slides/day1/trio_porcupine_fly_fish.jpg" width="700" alt="A porcupine, a close-up of a fruit fly's head, and a pufferfish, side by side">

What could we measure about each image that would help tell them apart?


## One feature: eye size relative to body

Plot a histogram of this feature across many labeled images. For two classes, a single dividing
line does a reasonable job:

<img src="assets/slides/day1/histogram_2class.png" width="420" alt="Histogram of porcupine and fruit fly images by eye-to-body-size ratio, showing two separable peaks">


## Adding a third class breaks it

<img src="assets/slides/day1/histogram_3class.png" width="420" alt="Histogram of porcupine, fruit fly, and fish images by eye-to-body-size ratio, showing three overlapping peaks that a single threshold cannot separate">

One feature, and even one dividing line, is no longer enough.


## Two features: a 2D feature space

Module 3 gave us many features, not just one. Add a second (say, *shape*), and each image becomes
a point in 2D space:

<img src="assets/slides/day1/feature_space_scatter.png" width="480" alt="Scatter plot of porcupine, fruit fly, and fish images plotted by shape versus eye-to-body size ratio, forming three separable clusters">

Now we can draw lines (or in higher dimensions, hyperplanes) that separate the classes. That's
exactly what both classifiers below do, using every feature Module 3 extracted, not just two.


The module below fits each classifier (support vector machine, then random forest) using
`sklearn`, including how to read a confusion matrix and where each one tends to get confused.

---


*Starting Module 4 fresh: the cell below clears all variables from the previous module before continuing, matching how these notebooks behaved when run standalone in their own kernel.*

In [ ]:
%reset -f

# Module 4 - Margin and Ensemble Classifiers

Before the popularization of deep learning, many applied automatic classification algorithms were variations on ensemble or margin classifiers. Both types of classifiers operate on pre-defined features. As we will see later, this is fundamentally different from neural networks which can learn directly from the images. For now, we will make use of the features pulled from SPC data in the last module.

In [ ]:
import numpy as np
import cv2
import skimage
from sklearn import ensemble
from sklearn import svm
from sklearn import preprocessing
import sys
import glob
import os
import random
import matplotlib.pyplot as plt
sys.path.insert(1, os.path.join(os.getcwd(),'computer-vision-workshop/utilities'))
from display_utils import make_confmat, tile_images

The important new toolkit we are importing here is *sklearn*, short for scikit-image. It contains most of the tools we will use to explore margin and ensemble classifiers.

## Module 4 – Importing features, dividing into training and test sets

For all the techinques discussed in the rest of this module, we will make use of the same features we computed before. We will also need to divide it into seperate sets for training and testing. 

In [ ]:
# load in the data. first get all the file paths
base_ptf = "/groups/cv-workshop/SPC_manual_labels_features/"
ptf = glob.glob(os.path.join(base_ptf, "*.csv"))

# initalize a dictionary for the data. This will contain all the file paths and the associated features
data = dict()

# we will also create a flag and a listto give the labels a numeric value
flag = 0
cls_names = []

for line in ptf:

    # read in the data, but skip the image path
    temp = np.genfromtxt(line, usecols=range(1,71),delimiter=",")
    
    # get the image path, making sure to specify that the data type is string
    temp_path = np.genfromtxt(line, usecols= [0], delimiter=",", dtype=str)
    
    # for now, we will ignore any classes with fewer than 10 samples
    if 10 < temp.shape[0]:
        
        # now we are creating a "nested dictionary." Each element is referenced by the image id and contains the features
        # and numeric class label
        for img, feats in zip(temp_path, temp):
            data[img] = {'features': feats, 'class': flag}
            
        # create a list of the names of the categories and the associated numbers
        name = line.split('/')[-1].split('_')[0]
        print("class", str(flag), ":", name, 
              ", num images:", str(temp.shape[0]))
        cls_names.append((flag, name))
        
        flag+=1

print("Total class:", str(flag), ", Total images:", str(len(data)))

We now have 38 classes, comprising a total of 20678 samples. The nested dictionary is how we will interact with the data. Each data point is identified by its image ID that is saved as a dictionary key.

Note that we are using python 3.6 which by default preserves the order of the dictionary. That is, the order of the key-value pairs will remain the same as how they were inserted into the dictionary no matter what. If for some reason you use an earlier version of python, be aware that the order may not be preserved. 

In [ ]:
# to get a list of all the dictionary keys use Python's built in list command and the dictionary method keys()
img_ids = list(data.keys())

print("the first sample is:", img_ids[0])

We will use this later to display images after they have been classifier. We can call up the information from that particular image by calling that key's associated values from the dictionary.

In [ ]:
# get the data related to a particular sample put the key in square brackets
# remember, the data is store as a dictionary itself. We can also print those keys in the same way

print(img_ids[0], "has two keys that can be referenced:", data[img_ids[0]].keys())

Now we can call the features and the class of that first sample.

In [ ]:
# to retrieve the class
print("the numeric class is:", data[img_ids[0]]['class'])

# and the features
print("the features are:",  data[img_ids[0]]['features'])

Now that we have all the data in the workspace, we need to divide it up for training and testing. That is we need to seperate out a subset of the training data to use as an independent set to assess how well the classifier is doing. 

To do the data dictionary needs to be randomized and split into a training and test set. Here we will use an 80-20 train-test split; 80% of the data will be used to train and 20% will be reserved for testing. 

In [ ]:
# to avoid copying the dictionary multiple times, we will randomize the list of keys (ie the image IDs) we made above.
random.shuffle(img_ids)

# print one out to double check
print("The new first entry is:", img_ids[0])

In [ ]:
# now we can split the list into training and test sets based on the number of entries
idx = 0.8*len(img_ids)

train_ids = img_ids[0:int(idx)]  # this will copy all the image ids from 0 to the 80% cut-off
test_ids = img_ids[int(idx)::]  # this will copy all the image ids from the cut-off to the end

# double check
print("cut off for 80-20 split:", str(int(idx)))
print("number of training images:", str(len(train_ids)))
print("nubmer of test images:", str(len(test_ids)))

With the data split by the image ID, we can select the training and test sets. 

In [ ]:
# to train it, feed in the features and labels

# pull out the features for the training data
# the next line uses "list comprehension" to pull out the feature vectors only from the trianing data
train_features = [data[line]['features'] for line in train_ids]
train_features = np.asarray(train_features)  # convert to an array

# retrieve the numeric classes of the training data
train_labels = [data[line]['class'] for line in train_ids]
train_labels = np.asarray(train_labels)  # convert to an array

# check to make sure these numbers are right. We expect the training features to be a matrix with 
# dimensions [n_images x n_features] and the training labels to be a matrix with dimensions [n_images x 1]
print("train features dim:", train_features.shape)
print("train labels dim:", train_labels.shape)

In [ ]:
# to test it, feed in the features and labels from the test data

# pull out the features for the test data
# the next line uses "list comprehension" to pull out the feature vectors only from the trianing data
test_features = [data[line]['features'] for line in test_ids]
test_features = np.asarray(test_features)  # convert to an array

# retrieve the numeric classes of the training data
test_labels = [data[line]['class'] for line in test_ids]
test_labels = np.asarray(test_labels)  # convert to an array

# check to make sure these numbers are right. We expect the test features to be a matrix with 
# dimensions [n_images x n_features] and the test labels to be a matrix with dimensions [n_images x 1]
print("test features dim:", test_features.shape)
print("test labels dim:", test_labels.shape)

The features and labels are now divided into training and test sets that we can use multiple times. The order of these will remain the same and correspond witht the order of the image IDs. 

## Module 4 – Feature standardization

It is good practice to standardize the features before feeding them into an ensemble or margin classifier. Here, standardization simply means making each feature look a Gaussian with zero mean and unit variance. SKLearn provides a class to fit a standardizer, save it, and apply it to both training and test data. 

In [ ]:
# invoke an instance of the standardizer class and fit it to the training features
scale_transform = preprocessing.StandardScaler().fit(train_features)

# The scale_transform instance stores all the information we need for the transformer
# print the mean of each feature
print("mean of first feature:", scale_transform.mean_[0])

Excellent. Now that the data is imported into the workspace in an organized way and the transformer prepared, we can begin training and testing classifiers. The same training and test data will be used for the both margin and ensemble classifiers. 

## Module 4 – Margin classifiers

A margin classifier seperates data in a space by assigning a distance between each point and the decision boundary. Imagine that we have just 2 features, $x_{1}$ and $x_{2}$, to seperate two classes. We can plot the points in a plane and find a line that seperates them.


<figure>
    <img src="https://upload.wikimedia.org/wikipedia/commons/b/b5/Svm_separating_hyperplanes_%28SVG%29.svg">
    <figcaption>
        Points and hyperplanes. Courtesy: ZackWeinberg, via Wikipedia
    </figcaption>
</figure>
        

The line labeled $H_{3}$ is the best linear discriminant of this data. 

A classic and widely used margin classifier is the *support vector machine* (SVM). SVMs search a space, definied by the features, to find the optimal seperating hyperplane (ie a plane in many dimensions). In the example above, it would iteratively try many lines such as $H_{1}$ and $H_{2}$ before eventually settling on a particular plane.

We will use the implimentation in SKLearn, svc -- a class that contains all functions need to train, test, and deplpoy a SVM. Do note however, that fitting a SVM is computationally expensive and scales quadratically. In other words, training an SVM with O(10k) samples becomes really time consuming and memory hungry.  

In [ ]:
# first create an instance of the SVM
svm_clf = svm.SVC(kernel='linear')

The kernerl parameter tells SKLearn how to seperate the data. A 'linear' kernerl will attempt to find linear decision boundaries. 

In [ ]:
# train the SVM (this may take a few minutes)
# the first parameter is the training features. Make sure to scale them!
# the second parameter is the training labels. These do not need to be scaled
svm_clf.fit(scale_transform.transform(train_features), train_labels)

Once trained, the classifier will show a bunch of information about the classifier. We can now apply it to new data to see how accurate it is.

In [ ]:
# run the new data through the classifier to get the mean accuracy
# the first parameter is the test data. Remember to scale it
acc_svm = svm_clf.score(scale_transform.transform(test_features), test_labels)

print("Linear SVM accuracy:", acc_svm)

We can visualize the accuracy of the classifier using a *confusion matrix* that compares the classifier labels to the true labels.  

In [ ]:
# To plot the confusion matrix, we need the predicitons from the classifier on an image-by-image basis.
svm_preds = svm_clf.predict(scale_transform.transform(test_features))

# feed the predictions and the true labels to a confusion matrix plotting utility
make_confmat(test_labels, svm_preds, acc_svm)

## Module 4 – Ensemble classifiers

Rather than relying on the results of a single classifer, ensemble classifier combine the results of many smaller classifiers. There are many ways of generating such a collection of computer classifiers. Here we will focus on the popular random forest (RFs) models. 

RFs build a collection of decision trees built from a random selection of the feature set. A single decision tree is a type of flow chart: each node in the tree is a test on a single feature and each branch denotes the outcome. A terminal node, or leaf, represents the tree's final classification.

A single decsion tree tends to overfit the data -- it become really good at representing the training data but does not generalize well to new data. RFs get around this by creating many trees, using a random subsample of features and training examples for each tree. A new sample is then fed into every tree and the results averaged at the end to come to a final decision.

To train a RF in python we will use the sklearn's ensemble methods.

<img src="assets/slides/day1/random_forest_diagram.png" width="600" alt="Diagram of a random forest: many decision trees each vote a class, combined by majority vote into a final class">


In [ ]:
# note we are only using a few of the RF parameters. There are many ways to modify this
rf_clf = ensemble.RandomForestClassifier(n_estimators=30, n_jobs=8, verbose=1)

We have not trained the classifier yet. The code above defines an instance *rf_clf* of the class *RandomForestClassifier*. The parameters we added define a few things:

* n_estimators is the number of trees. For now we are just using 30
* n_jobs parallalizes the process. It subdivides the process out to some number of cores. This speeds up training and is limited by the hardware you are working on. 
* verbose just tells sklearn they we want feedback as it is training. 

There are loads of other parameters that can change how the classifier behaves. For our purposes, mostly using defaults will suffice. Note that we are not exlicitly defining the number of features the will be used in each random tree. The default as set by sklearn is $\sqrt{n\_features}$. This is generally a good rule of thumb.

To train the classifier we need to give it data. This is done with the *fit* method of the *RandomForestClassifier*.

In [ ]:
# plug into the fit method. this step might take a little while.
# much as with the SVM, be sure to scale the training features.
rf_clf.fit(scale_transform.transform(train_features), train_labels)

All done! The classifier is trained. Now we can test it on the independent set that it has not seen yet. 

In [ ]:
# plug test data into the trained classifier
acc = rf_clf.score(scale_transform.transform(test_features), test_labels)

print(acc, '% correct')

Not too bad. But we need a way of visualizing what classes it had trouble with. 

In [ ]:
# get the labels for the test set from the classifier
preds = rf_clf.predict(scale_transform.transform(test_features))

# make a confusion matrix
make_confmat(test_labels, preds, acc)

In the ideal case, this matrix would be purely diagonal. The class at index 17 is particularly interesting. There seems to be a lot of data going there. Which one is it?

In [ ]:
cls_names[17]

Aggregate Mix is a mishmash class. It seems like one that the classifier could easily confuse. It is also a big class (~2k images) relative to some of the small ones. This means there is some variability in that might not have been captured in the training data for some of the spase classes. 

With the dictonaries, we can make a list of all the images labeled as "Aggregate Mix" and see a few to get a sense of what was confusing the classifier.

In [ ]:
# get the indicies of images that were labeled as the "Aggregate Mix" class.
# nonzero creates a list of booleans - True when the condition is met, false otherwise
[mask, ] = np.nonzero(preds == 17)

# convert the ndarray to a list for indexing
mask = list(mask)

# actually get the associated image ids
labeled_as_skinny = [test_ids[item] for item in mask]
len(labeled_as_skinny)

In [ ]:
# create a list of the first 20 images labeled at skinny mix
# first we need to retrieve the true classes from the image IDs
true_numeric = [test_labels[item] for item in mask[0:20]]

# now convert this to a list of strings with the true label
true_str = [cls_names[item][1] for item in true_numeric]

# now make a list of file paths by combining the image ID, label and base path
labeled_imgs = [os.path.join(base_ptf, "../SPC_manual_labels", item[0], item[1]) 
                for item in zip(true_str, labeled_as_skinny[0:20])]

# now tile the images using the image tiling utility
tiled = tile_images(labeled_imgs, [2, 10])

# display it
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xticks([])
ax.set_yticks([])
ax.imshow(tiled)

It is very hard to tell some of these apart, even for a human. The classifier is mapping many of these skinny objects to this mixed class. 

## Module 4 – Excercises

Complete the following for extra practice.

1. Train and test a new SVM with a Radial Basis Function (RBF) kernel.
A RBF kernel computes nonlinear decision boundaries in the feature space. This can be an effective way of representing more complex data types. Mathematically, this "kernel trick" projects the data to a higher dimensional space where it is linearly seperable.

In [ ]:
# instantiate a new classifier with the kernel set to 'rbf'
svm_rbf = svm.SVC()

# train it on the same training data as above, being careful to scale the features.
svm_rbf.fit()

# compute the accuracy on the test dat
rbf_acc = svm_rbf.score()

# print out the result
print("SVM with RBF mean accuracy=", rbf_acc)

How does this classifier compare to the linear SVM? 

Be aware that there are many SVM parameters that we are not covering here. The RBF, for example, has two "hyperparameters", the $\gamma$ term in particular, that must be tuned and tested. This is usually done with a process called "cross-validation" to optimize the best parameters for the classifier. 

2. Train a series of RFs with progressively more trees and evaluate the performance.
The number of trees used for a random forest is an important hyperparamter. Intuitively, having more trees seems like it would lead to better results. Try it out and see!

In [ ]:
# write a for-loop to iteratively train an RF with the following number of trees.
num_trees = [5, 10, 20, 50, 100, 150, 200, 250]

# initalize a list to hold all the scores
acc_out = []

# the for-loop
for tr in num_trees:
    
    # instantiate the classifier
    rf_temp = 
    
    # train it with scaled features
    rf_temp.fit()
    
    # test the trained classifier 
    acc_temp = rf_temp
    
    # save the accuracy
    acc_out.append(acc_temp)
    
# plot the output
fig, ax = plt.subplots(figsize=(14,7))
ax.plot(num_trees, acc_out)
ax.xlabel('number of trees')
ax.ylabel('mean test accuracy')

What is the pattern? What seems to be a reasonable number of trees?

In this example, it does not much matter since training takes such a short period of time. But with more training images, this can be a lengthy process that makes any computational savings worthwhile.